In [ ]:
using Pkg
Pkg.activate("..")
using MLDatasets, DataFrames, CSV, LinearAlgebra, MLDataUtils, Statistics, Plots
housing = select(CSV.read("housing.csv", DataFrame), Not(:ocean_proximity))

*Question 3*


In [100]:
"""
estimate values using kernel regression: 
X: The design matrix 
x: the points where we wish to predict 
y: the response variables
h: the smoothing parameter for the kernel function
note: this uses a gaussian kernel
"""
function m(X::AbstractVecOrMat, x::AbstractVecOrMat, y::AbstractVector, h::AbstractFloat, kernel::Function)
    
    norms = kernel(x, X, h)
    multiplied_value = norms * y
    valueSums = sum(norms, dims = 2)
    return multiplied_value./valueSums
    
end
"""
returns the kernel value based on a gaussian norm
x: the points to be estimated
X: the design matrix

return: a vector of size n (for n samples in x)
"""
function gaussianKernel(train::AbstractVecOrMat, test::AbstractVecOrMat, h::AbstractFloat)
    returnValue = zeros(size(train,1), size(test,1))
    for i in 1:size(test, 1)
        row = test[i,:]
        d2 = -norm.(eachrow(train .- row)).^2/(2h^2)
        returnValue[:, i] = exp.(d2)
        
    end
    return returnValue
end


"""
return the MSE based on a ground truth and estimates
"""
function MSE(Y_est::AbstractVector, Y::AbstractVector)
    
    mean.(norm(Y_est-Y).^2)

end
(train, test) = splitobs(dropmissing!(housing), at = .8)
y_train = train[:,"median_house_value"]
X_train = Matrix(select(train, Not(:median_house_value)))
y_test =  test[:,"median_house_value"]
X_test = Matrix(select(test, Not(:median_house_value)))
h= [0.025, 0.05, 0.1, 0.2, 0.4, 0.8]

estimates = m.(Ref(X_train[:,8]),Ref(X_test[:,8]), Ref(y_train), h, Ref(gaussianKernel))



6-element Vector{Matrix{Float64}}:
 [197015.16879609373; 178826.39344527872; … ; 115439.18082220982; 140464.70087702523;;]
 [196476.9785313548; 184483.86388149482; … ; 117580.95488758573; 139882.29855370274;;]
 [195909.2582211072; 185365.94828236397; … ; 117764.11105531256; 138945.5153673293;;]
 [195261.17723950825; 184259.4283445607; … ; 118508.91249451773; 138824.16980523366;;]
 [193772.15247155871; 182622.55960373793; … ; 123384.77958401837; 141517.674803691;;]
 [188804.44431250542; 179493.77507010248; … ; 136855.75256486342; 150296.99475320228;;]

In [105]:
display(MSE(vec(estimates[1]), y_test))
display(MSE(vec(estimates[2]), y_test))
display(MSE(vec(estimates[3]), y_test))
display(MSE(vec(estimates[4]), y_test))
display(MSE(vec(estimates[5]), y_test))
display(MSE(vec(estimates[6]), y_test))


2.6812851941812195e13

2.6709817661553477e13

2.6632504926895434e13

2.663425565814974e13

2.693694742320517e13

2.8496610806032777e13